In [ ]:
# Set up for plotting
# Params for clean and minimalistic plots

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams.update({
    #'figure.figsize': (6*2, 2*4.5),
    'font.size': 16.0,
    'font.family': 'serif',
    'font.serif': 'Palatino',
    'axes.titlesize': 'medium',
    'figure.titlesize': 'large',
    'legend.fontsize': 'medium',
    # dpi for high-res output
    'figure.dpi': 100,
    'savefig.dpi': 300,
    # Tight layout by default
    'figure.autolayout': True,
    'text.usetex': True,
    'text.latex.preamble': r"\usepackage{amsmath}\usepackage{amssymb}\usepackage{siunitx}[=v2]",
})

from pathlib import Path

PLOT_ROOT = Path.cwd() / "Plots"

In [ ]:
import sys
import numpy as np
import pandas as pd
import geopandas as gpd
import statsmodels.api as sm

sys.path.insert(0, str(Path.cwd()))
from helpers import (
    load_tt_matrix,
    load_blocks_with_bezirk,
    load_commuting_data,
    normalize_commuting_columns,
    build_block_bezirk_map,
    aggregate_tt_to_bezirke,
    build_gravity_df,
    run_ols_gravity,
    run_poisson_gravity,
    run_gamma_gravity,
    partial_out_fe,
    format_gravity_latex_table,
)

# ── Paths ─────────────────────────────────────────────────────────────────────
REPO_ROOT = Path.cwd().parent                    # QSE-Tutorial/
DATA_ROOT  = REPO_ROOT / "Data"

TTM_PATH  = Path.cwd() / "TTM" / "sample_travel_time_matrix.parquet"
BLOCKS_SHP = DATA_ROOT / "Shapefiles-2022" / "Berlin" / "Berlin4matlab-ETRS.shp"

# ── Set COMMUTING_DATA_PATH to your local copy of the ARSW replication data ──
# Expected: CSV, DTA, or XLSX with one row per bilateral Bezirk pair.
# Columns: residence district ID (int or str 1–12), workplace district ID, commuter count.
# Example: Path("/path/to/arsw_teaching/data/commuting_bezirke_2008.csv")
COMMUTING_DATA_PATH = Path("ARSW2015-toolkit") / "data" / "commuting_flows.dta"

# Correct PLOT_ROOT to match existing lowercase directory
PLOT_ROOT = Path.cwd() / "plots"
PLOT_ROOT.mkdir(exist_ok=True)

# ── Column names in the commuting data ────────────────────────────────────────
# Adjust these three variables to match your actual file's column names.
RES_COL   = "bezirk_res"    # residence district column
WORK_COL  = "bezirk_work"   # workplace district column
FLOW_COL  = "n_commuters"   # commuter count column

## Task 1(a) — Commuting Gravity Regression

**Structural equation (ARSW 2015, eq. 23/25).** The ARSW model's bilateral commuting
probability $\pi_{ij}$ (worker living in block $i$ working in block $j$) satisfies

$$\ln \pi_{ij} = -\nu\,\tau_{ij} + \vartheta_i + \varsigma_j$$

where $\nu \equiv \varepsilon\kappa$ is the semi-elasticity of commuting flows with
respect to travel time, $\vartheta_i$ are residence fixed effects absorbing $\{B_i, T_i, Q_i\}$,
and $\varsigma_j$ are workplace fixed effects absorbing $\{w_j, E_j\}$.
Commuting costs are iceberg: $d_{ij} = e^{\kappa\tau_{ij}}$.

Since commuting data are available at the Bezirk (district) level ($I,J \in \{1,\ldots,12\}$),
we estimate the aggregated version (ARSW eq. 25):

$$\ln \pi_{IJ} = -\nu\,\bar{\tau}_{IJ} + \vartheta_I + \varsigma_J + e_{IJ}$$

where $\bar{\tau}_{IJ} = |I|^{-1}|J|^{-1}\sum_{i\in I}\sum_{j\in J}\tau_{ij}$
is the mean block-level travel time between Bezirke $I$ and $J$ from our computed matrix.
ARSW verify that this Jensen approximation (mean of logs $\approx$ log of mean) is
quantitatively negligible ($\hat{\nu}_{\text{block}} = 0.07 \approx \hat{\nu}_{\text{district}} = 0.0726$).

The four specifications replicate **ARSW Table III**: OLS (all 144 pairs),
OLS ($\geq 10$ commuters, 122 pairs), Poisson PML, Gamma PML.

In [ ]:
# ── 1. Travel time matrix ─────────────────────────────────────────────────────
print("Loading travel time matrix...")
tt_matrix = load_tt_matrix(TTM_PATH)
print(f"  Matrix shape : {tt_matrix.shape}")
print(f"  Index sample : {list(tt_matrix.index[:3])}")
print(f"  Value range  : [{tt_matrix.values[tt_matrix.values > 0].min():.1f}, "
      f"{np.nanmax(tt_matrix.values):.1f}] minutes")

# ── 2. Berlin blocks with Bezirk assignment ───────────────────────────────────
print("\nLoading Berlin block shapefile...")
blocks_gdf = load_blocks_with_bezirk(BLOCKS_SHP)
print(f"  Blocks loaded : {len(blocks_gdf):,}")
print(f"  Bezirke found : {sorted(blocks_gdf['BEZIRK'].unique())}")
print(f"  Columns       : {list(blocks_gdf.columns)}")

# Build centroid_id → BEZIRK mapping
block_bezirk = build_block_bezirk_map(blocks_gdf, bezirk_col='BEZIRK')
print(f"\n  Block→Bezirk map: {len(block_bezirk):,} entries, "
      f"{block_bezirk.nunique()} unique Bezirke")

# ── 3. Commuting flow data ────────────────────────────────────────────────────
print("\nLoading commuting flow data...")
flows_raw = load_commuting_data(COMMUTING_DATA_PATH)
print(f"  Rows: {len(flows_raw)}, Columns: {list(flows_raw.columns)}")
print(flows_raw.head())

In [ ]:
# ── Aggregate N×N block matrix → 12×12 Bezirke matrix ─────────────────────────
print("Aggregating block-level travel time matrix to Bezirke level...")
print("  (τ̄_IJ = mean of τ_ij for blocks i∈I, j∈J)")

tt_bezirke = aggregate_tt_to_bezirke(tt_matrix, block_bezirk)

print(f"\n  Bezirke-level matrix: {tt_bezirke.shape}")
print(f"  Bezirke: {sorted(tt_bezirke.index.tolist())}")
print(f"\n  Travel time summary (minutes):")
vals = tt_bezirke.values[~np.eye(len(tt_bezirke), dtype=bool)]  # off-diagonal
print(f"    Off-diagonal mean : {vals.mean():.1f}")
print(f"    Off-diagonal min  : {vals.min():.1f}")
print(f"    Off-diagonal max  : {vals.max():.1f}")

print("\nBezirke-level travel time matrix (minutes):")
display(tt_bezirke.round(1))

In [ ]:
# ── Normalize column names in commuting data ──────────────────────────────────
flows = normalize_commuting_columns(flows_raw, RES_COL, WORK_COL, FLOW_COL)

print(f"Total commuters in dataset: {flows['n_comm'].sum():.0f}")
print(f"Bilateral pairs: {len(flows)}")
print(f"Pairs with ≥10 commuters: {(flows['n_comm'] >= 10).sum()}")

# ── Build estimation datasets ──────────────────────────────────────────────────
df_full    = build_gravity_df(flows, tt_bezirke, min_n=None)   # all pairs (Col 1)
df_min10   = build_gravity_df(flows, tt_bezirke, min_n=10)     # ≥10 commuters (Cols 2–4)

print(f"\nDataset sizes:")
print(f"  Full sample (Col 1)     : {len(df_full)} pairs")
print(f"  ≥10 commuters (Cols 2-4): {len(df_min10)} pairs")
print(f"\nTravel time range in estimation sample:")
print(f"  Full: [{df_full['tau'].min():.1f}, {df_full['tau'].max():.1f}] min")
print(f"\nln(π) range:")
print(f"  Full: [{df_full['ln_pi'].min():.3f}, {df_full['ln_pi'].max():.3f}]")

In [ ]:
# ── Four gravity specifications (ARSW Table III) ───────────────────────────────
print("Running gravity regressions...")

res_1 = run_ols_gravity(df_full)                   # (1) OLS, all pairs
res_2 = run_ols_gravity(df_min10)                  # (2) OLS, ≥10 commuters
res_3 = run_poisson_gravity(df_min10)              # (3) Poisson PML
res_4 = run_gamma_gravity(df_min10)                # (4) Gamma PML

results = [res_1, res_2, res_3, res_4]

# Print individual summaries (for inspection)
for i, (r, label) in enumerate(zip(results, ['OLS full', 'OLS ≥10', 'Poisson PML', 'Gamma PML']), 1):
    nu_hat = r.params['tau']
    se     = r.bse['tau']
    pval   = r.pvalues['tau']
    r2     = getattr(r, 'rsquared', None)
    n_obs  = int(r.nobs)
    r2_str = f"R² = {r2:.4f}" if r2 is not None else "R² = n/a"
    print(f"  ({i}) {label:15s}  ν̂ = {nu_hat:.4f}  SE = {se:.4f}  p = {pval:.4f}  {r2_str}  N = {n_obs}")

In [ ]:
# ── ARSW Table III replica ─────────────────────────────────────────────────────
ns     = [len(df_full), len(df_min10), len(df_min10), len(df_min10)]
r2s    = [
    res_1.rsquared,
    res_2.rsquared,
    None,   # Poisson PML — no conventional R²
    None,   # Gamma PML
]
estimators     = ["OLS", "OLS", "Poisson PML", "Gamma PML"]
min_comm_flags = [False, True, True, True]

latex_table = format_gravity_latex_table(
    results=results,
    ns=ns,
    r2s=r2s,
    estimators=estimators,
    min_comm_flags=min_comm_flags,
)

print("=" * 72)
print("READY-TO-COPY LaTeX TABLE (ARSW Table III replica)")
print("=" * 72)
print(latex_table)
print("=" * 72)

In [ ]:
# ── Figure 4A equivalent: residualized gravity scatter ────────────────────────
# Uses the Column (2) sample (≥10 commuters) and OLS spec to match ARSW.
# FWL: partial out two-way FEs from both ln(π) and τ,
# then scatter the residuals. Slope of fitted line = −ν̂.

df_plot = df_min10.copy()
df_plot["ln_pi_resid"] = partial_out_fe(df_plot["ln_pi"], df_plot["res_fe"], df_plot["work_fe"])
df_plot["tau_resid"]   = partial_out_fe(df_plot["tau"],   df_plot["res_fe"], df_plot["work_fe"])

nu_hat_col2 = res_2.params["tau"]
x_fit = np.linspace(df_plot["tau_resid"].min(), df_plot["tau_resid"].max(), 200)
y_fit = nu_hat_col2 * x_fit   # fitted line through origin (after demeaning)

fig, ax = plt.subplots(figsize=(6, 5))

ax.scatter(
    df_plot["tau_resid"],
    df_plot["ln_pi_resid"],
    color="#2c6fad",
    alpha=0.75,
    s=45,
    zorder=3,
    label=r"Bez\-irke pairs ($\geq 10$ commuters)",
)

ax.plot(
    x_fit,
    y_fit,
    color="#c0392b",
    linewidth=1.8,
    zorder=4,
    label=rf"OLS fit: slope $= {nu_hat_col2:.4f}$ ($= -\hat{{\nu}}$)",
)

ax.axhline(0, color="black", linewidth=0.5, linestyle="--", alpha=0.4)
ax.axvline(0, color="black", linewidth=0.5, linestyle="--", alpha=0.4)

ax.set_xlabel(r"Travel time residual $\tilde{\tau}_{IJ}$ (minutes)", labelpad=8)
ax.set_ylabel(r"$\ln \pi_{IJ}$ residual $\tilde{\pi}_{IJ}$", labelpad=8)
ax.set_title(
    r"Gravity fit: $\ln \pi_{IJ}$ on $\tau_{IJ}$ (two-way FE partialled out)",
    pad=10,
)
ax.legend(frameon=True, framealpha=0.9, fontsize=12)

# Annotate with ν̂ and R² from Column (2)
r2_col2 = res_2.rsquared
ax.annotate(
    rf"$\hat{{\nu}} = {abs(nu_hat_col2):.4f}$, $R^2 = {r2_col2:.3f}$",
    xy=(0.05, 0.08),
    xycoords="axes fraction",
    fontsize=12,
    color="#2c3e50",
)

save_path = PLOT_ROOT / "fig4a_gravity_fwl_scatter.pdf"
fig.savefig(save_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {save_path}")

In [ ]:
# ── Summary ───────────────────────────────────────────────────────────────────
print("══════════════════════════════════════════════════════════════════════")
print("  Task 1(a) — Commuting Gravity Regression Results")
print("══════════════════════════════════════════════════════════════════════")
print()
print(f"  Estimated ν̂ = εκ  (semi-elasticity of commuting w.r.t. travel time)")
print()
print(f"  (1) OLS, all pairs       :  ν̂ = {abs(res_1.params['tau']):.4f}  "
      f"(SE = {res_1.bse['tau']:.4f},  R² = {res_1.rsquared:.3f},  N = {len(df_full)})")
print(f"  (2) OLS, ≥10 commuters   :  ν̂ = {abs(res_2.params['tau']):.4f}  "
      f"(SE = {res_2.bse['tau']:.4f},  R² = {res_2.rsquared:.3f},  N = {len(df_min10)})")
print(f"  (3) Poisson PML          :  ν̂ = {abs(res_3.params['tau']):.4f}  "
      f"(SE = {res_3.bse['tau']:.4f},  N = {len(df_min10)})")
print(f"  (4) Gamma PML            :  ν̂ = {abs(res_4.params['tau']):.4f}  "
      f"(SE = {res_4.bse['tau']:.4f},  N = {len(df_min10)})")
print()
print(f"  ARSW (2015) benchmark    :  ν̂ = 0.0702  (Table III, Col 2)")
print()
print(f"  Each extra minute of travel time reduces commuting probability by ~{abs(res_2.params['tau'])*100:.1f}%.")
print()
print(f"  → This ν̂ = εκ is the input to Step 1(b): use optimepsilon_TD86.m")
print(f"    to disentangle ε and κ from the wage dispersion in West Berlin 1986.")
print("══════════════════════════════════════════════════════════════════════")